TASK1

In [ ]:
import pandas as pd
import numpy as np
import pyarrow
import json
import random
import time
import os
import copy

In [ ]:
dataset_a = [
    {"username": "alice_99", "email": "alice@example.com", "age": 25, "signup_date": "2025-01-01"},
    {"username": "bob_builder", "email": "bob@gmail.com", "age": 42, "signup_date": "2025-01-02"},
    {"username": "charlie_win", "email": "charlie_at_gmail.com", "age": 30, "signup_date": "2025-01-03"}, 
    {"username": "daria_dev", "email": "daria@company.org", "age": "twenty-eight", "signup_date": "2025-01-04"}, 
    {"username": "", "email": "ghost@void.com", "age": 19, "signup_date": "2025-01-05"},
    {"username": "eve_online", "email": "eve@proton.me", "age": 35, "signup_date": "2025-01-06"},
    {"username": "frank_transport", "email": "frank@delivery.com", "age": 50, "signup_date": "2025-01-07"},
    {"username": "grace_hopper", "email": "grace@navy.mil", "age": 85, "signup_date": "2025-01-08"}
]

dataset_b = [
    {"timestamp": "2025-03-01 10:00:01", "service": "Auth", "level": "INFO", "message": "User logged in"},
    {"timestamp": "2025-03-01 10:00:05", "service": "Payment", "level": "ERROR", "message": "Gateway timeout"},
    {"timestamp": "2025-03-01 10:00:10", "service": "Inventory", "level": "WARNING", "message": "Low stock on SKU-102"},
    {"timestamp": "2025-03-01 10:01:00", "service": "Auth", "level": "INFO", "message": "Token refreshed"},
    {"timestamp": "2025-03-01 10:02:15", "service": "Payment", "level": "INFO", "message": "Transaction successful"},
    {"timestamp": "2025-03-01 10:05:00", "service": "Inventory", "level": "INFO", "message": "Stock updated"},
    {"timestamp": "2025-03-01 10:06:20", "service": "Auth", "level": "WARNING", "message": "Failed login attempt"},
    {"timestamp": "2025-03-01 10:07:45", "service": "Payment", "level": "ERROR", "message": "Card declined"},
    {"timestamp": "2025-03-01 10:08:12", "service": "Inventory", "level": "INFO", "message": "Report generated"},
    {"timestamp": "2025-03-01 10:10:00", "service": "Auth", "level": "INFO", "message": "User logged out"}
]

dataset_c = [
    {"sku": "SKU-001", "product_name": "Mechanical Keyboard", "quantity_in_stock": 50, "warehouse_location": "A1", "last_updated": "2025-02-28"},
    {"sku": "SKU-002", "product_name": "Gaming Mouse", "quantity_in_stock": 120, "warehouse_location": "B2", "last_updated": "2025-02-28"},
    {"sku": "SKU-003", "product_name": "Ultrawide Monitor", "quantity_in_stock": 15, "warehouse_location": "A3", "last_updated": "2025-03-01"},
    {"sku": "SKU-004", "product_name": "USB-C Hub", "quantity_in_stock": 200, "warehouse_location": "C1", "last_updated": "2025-03-01"},
    {"sku": "SKU-005", "product_name": "Desk Mat", "quantity_in_stock": 80, "warehouse_location": "B1", "last_updated": "2025-03-02"},
    {"sku": "SKU-006", "product_name": "Webcam", "quantity_in_stock": 45, "warehouse_location": "A2", "last_updated": "2025-03-02"}
]

Dataset A (User Input): This represents raw user-generated data. In production, expect typos, malicious injections, and missing fields. Validation must include strict type checking (e.g., ensuring age is an integer) and regex patterns for emails.

Dataset B (System-Generated): This is observability data (logs). In production, the main issues are high volume, inconsistent formatting across services, and missing timestamps. Validation involves parsing strings into structured formats and filtering for severity levels.

Dataset C (Internal Database): This represents system of record data. It is usually the cleanest, but can suffer from "drift" (outdated timestamps) or synchronization errors between warehouses. Validation should ensure referential integrity and non-negative stock values.

In [ ]:
def validate_registrations(entries):
    valid, invalid = [], []
    for entry in entries:
        try:
            is_valid = (
                bool(entry['username']) and 
                "@" in entry['email'] and 
                int(entry['age']) > 0
            )
            if is_valid: valid.append(entry)
            else: invalid.append(entry)
        except (ValueError, TypeError):
            invalid.append(entry)
    return valid, invalid

valid, invalid = validate_registrations(dataset_a)
print(f"Valid: {len(valid)}, Invalid: {len(invalid)}")

TASK2

In [ ]:
rides = pd.DataFrame({
    'ride_id': range(1, 501),
    'driver_id': np.random.randint(1000, 1051, 500),
    'distance_km': np.random.uniform(1.0, 30.0, 500),
    'fare_usd': np.random.uniform(5.0, 75.0, 500),
    'ride_type': np.random.choice(["standard", "premium", "pool"], 500),
    'city': np.random.choice(["Berlin", "Seoul", "Nairobi", "Toronto", "Lima"], 500),
    'timestamp': pd.date_range("2025-01-01", periods=500, freq='min')
})

In [ ]:
rides.to_csv("rides.csv", index=False)
rides.to_json("rides.json", orient="records", lines=True)
rides.to_parquet("rides.parquet")

sizes = {f: os.path.getsize(f) / 1024 for f in ["rides.csv", "rides.json", "rides.parquet"]}
for f, s in sizes.items(): print(f"{f}: {s:.2f} KB")
print(f"Parquet is {sizes['rides.csv']/sizes['rides.parquet']:.1f}x smaller than CSV.")

Most Faithful: Parquet preserved all types (integers, floats, and timestamps) perfectly. CSV lost the datetime type (converted to string).

Smallest: Parquet (due to columnar compression).

Usage Choice: Use CSV for human readability/quick inspection. Use JSON for web APIs and nested data. Use Parquet for big data storage and ML pipelines where performance and schema preservation are critical.

TASK3

In [ ]:
np.random.seed(1)
N_ROWS, N_COLS = 200_000, 20
col_names = [f"feature_{i}" for i in range(N_COLS)]
big_df = pd.DataFrame(np.random.randn(N_ROWS, N_COLS), columns=col_names)

In [ ]:
t0 = time.perf_counter()
for _ in range(100): _ = big_df["feature_0"].mean()
t_col_pd = (time.perf_counter() - t0) / 100 * 1000  # ms

In [ ]:
t0 = time.perf_counter()
total = sum(big_df.iloc[i]["feature_0"] for i in range(10_000))
t_row_pd = (time.perf_counter() - t0) * 1000

print(f"Pandas column mean (200k rows) : {t_col_pd:.2f} ms")
print(f"Pandas row iter   (10k rows)   : {t_row_pd:.1f} ms  (~{t_row_pd/t_col_pd:.0f}x slower)")

In [ ]:
arr = big_df.to_numpy()

t0 = time.perf_counter()
for _ in range(100): _ = arr[:, 0].mean()
t_col_np = (time.perf_counter() - t0) / 100 * 1000

t0 = time.perf_counter()
total = sum(arr[i, 0] for i in range(10_000))
t_row_np = (time.perf_counter() - t0) * 1000

In [ ]:
print(f"\nNumPy column mean (200k rows)  : {t_col_np:.2f} ms")
print(f"NumPy row iter    (10k rows)   : {t_row_np:.1f} ms  (~{t_row_np/t_col_np:.0f}x slower)")

Why is pandas column access faster than row iteration?
Pandas stores each column as a contiguous NumPy array.
df["feature_0"].mean() runs in optimized C code over a single, cache-friendly memory block with no Python overhead.

Row iteration (df.iloc[i]) crosses the Python/C boundary every time, builds a new Series object, and gathers data from multiple columns. This is cache-unfriendly and much slower.

Why does the gap narrow with NumPy?
NumPy arrays are row-major (C-contiguous). Rows are stored contiguously, columns are strided.

So column access (arr[:, 0]) is less cache-efficient, while row access (arr[i]) is contiguous. The Python loop is still slow, but the performance gap shrinks.

Takeaway:
Pandas is optimized for column-wise analytics.
NumPy (row-major) is better when processing all features of one sample at once.
Choose the representation based on your access pattern.

TASK4

In [ ]:
np.random.seed(42)

df_large = pd.DataFrame(np.random.randn(100000, 5), columns=[f'feat_{i}' for i in range(5)])
df_large['category'] = np.random.choice(["alpha", "beta", "gamma"], 100000)

df_large.to_csv("large.csv", index=False)
df_large.to_parquet("large.parquet")

In [ ]:
sizes_large = {f: os.path.getsize(f) / 1024 for f in ["large.csv", "large.parquet"]}
print(f"Parquet is {sizes_large['large.csv']/sizes_large['large.parquet']:.1f}x smaller than CSV.")

In [ ]:
pd.read_csv('large.csv').dtypes

In [ ]:
pd.read_parquet('large.parquet').dtypes

In [ ]:

start = time.time(); pd.read_csv("large.csv"); csv_time = time.time() - start
start = time.time(); pd.read_parquet("large.parquet"); pq_time = time.time() - start

print(f"CSV Read: {csv_time:.4f}s | Parquet Read: {pq_time:.4f}s")

In [ ]:
%time pd.read_parquet("large.parquet", columns=["category"])

In [ ]:
%time pd.read_parquet("large.parquet")

Parquet is a columnar storage format.  
This means each column is stored separately on disk.

Because of this structure, Parquet can load only the requested column
without reading the entire dataset. This significantly reduces disk I/O
and improves performance.

CSV is a row-based plain text format.  
Even when selecting one column with `usecols`, the file must still be
read line by line and parsed completely before extracting that column.

Therefore:

- Parquet supports true column pruning.
- CSV must scan the entire file.
- Parquet is much more efficient for large datasets when only a few columns are needed.

TASK5

In [ ]:
flat_books = pd.DataFrame([
    {"title": "Sapiens",               "author": "Yuval Noah Harari", "format": "Paperback", "publisher_name": "Harper Perennial", "publisher_country": "USA", "price": 16.99},
    {"title": "Sapiens",               "author": "Yuval Noah Harari", "format": "E-book",    "publisher_name": "Harper Perennial", "publisher_country": "USA", "price": 12.99},
    {"title": "Dune",                  "author": "Frank Herbert",      "format": "Paperback", "publisher_name": "Ace Books",        "publisher_country": "USA", "price": 14.99},
    {"title": "Dune",                  "author": "Frank Herbert",      "format": "E-book",    "publisher_name": "Ace Books",        "publisher_country": "USA", "price":  9.99},
    {"title": "The Midnight Library",  "author": "Matt Haig",          "format": "Paperback", "publisher_name": "Canongate Books", "publisher_country": "UK",  "price": 13.49},
    {"title": "The Midnight Library",  "author": "Matt Haig",          "format": "E-book",    "publisher_name": "Canongate Books", "publisher_country": "UK",  "price":  8.99},
    {"title": "Atomic Habits",         "author": "James Clear",        "format": "Paperback", "publisher_name": "Avery",            "publisher_country": "USA", "price": 18.00},
    {"title": "Atomic Habits",         "author": "James Clear",        "format": "E-book",    "publisher_name": "Avery",            "publisher_country": "USA", "price": 13.00},
    {"title": "Norwegian Wood",        "author": "Haruki Murakami",    "format": "Paperback", "publisher_name": "Vintage",          "publisher_country": "UK",  "price": 12.50},
    {"title": "Kafka on the Shore",    "author": "Haruki Murakami",    "format": "E-book",    "publisher_name": "Vintage",          "publisher_country": "UK",  "price":  9.50},
    {"title": "The Pragmatic Programmer","author":"David Thomas",       "format": "Paperback", "publisher_name": "Addison-Wesley",  "publisher_country": "USA", "price": 49.99},
    {"title": "Clean Code",            "author": "Robert C. Martin",   "format": "Paperback", "publisher_name": "Prentice Hall",   "publisher_country": "USA", "price": 39.99},
])


In [ ]:

publishers = flat_books[["publisher_name","publisher_country"]].drop_duplicates().reset_index(drop=True)
publishers.insert(0, "publisher_id", publishers.index + 1)
books = flat_books.merge(publishers, on=["publisher_name","publisher_country"])[
    ["title","author","format","publisher_id","price"]]

In [ ]:

rejoined = books.merge(publishers, on="publisher_id")[flat_books.columns]
assert rejoined.sort_values(["title","format"]).reset_index(drop=True).equals(
       flat_books.sort_values(["title","format"]).reset_index(drop=True))
print("Normalized round-trip verified ✓")

In [ ]:
doc_books = [
    {"title":"Sapiens",      "author":"Yuval Noah Harari","publisher":{"name":"Harper Perennial","country":"USA"},"editions":[{"format":"Paperback","price":16.99},{"format":"E-book","price":12.99}]},
    {"title":"Dune",         "author":"Frank Herbert",     "publisher":{"name":"Ace Books","country":"USA"},        "editions":[{"format":"Paperback","price":14.99},{"format":"E-book","price":9.99}]},
    {"title":"The Midnight Library","author":"Matt Haig",  "publisher":{"name":"Canongate Books","country":"UK"},  "editions":[{"format":"Paperback","price":13.49},{"format":"E-book","price":8.99}]},
    {"title":"Atomic Habits","author":"James Clear",       "publisher":{"name":"Avery","country":"USA"},            "editions":[{"format":"Paperback","price":18.00},{"format":"E-book","price":13.00}]},
    {"title":"Norwegian Wood","author":"Haruki Murakami",  "publisher":{"name":"Vintage","country":"UK"},           "editions":[{"format":"Paperback","price":12.50}]},
    {"title":"Kafka on the Shore","author":"Haruki Murakami","publisher":{"name":"Vintage","country":"UK"},         "editions":[{"format":"E-book","price":9.50}]},
    {"title":"The Pragmatic Programmer","author":"David Thomas","publisher":{"name":"Addison-Wesley","country":"USA"},"editions":[{"format":"Paperback","price":49.99}]},
    {"title":"Clean Code",   "author":"Robert C. Martin",  "publisher":{"name":"Prentice Hall","country":"USA"},   "editions":[{"format":"Paperback","price":39.99}]},
]

In [ ]:
author = "Haruki Murakami"
print("--- Q1: books by", author)
print("[Flat]      \n", flat_books[flat_books.author==author][["title","format","price"]].to_string(index=False))
print("[Normalized]\n", books[books.author==author][["title","format","price"]].to_string(index=False))
print("[Document]  ")
for b in doc_books:
    if b["author"]==author:
        for e in b["editions"]: print(f"  {b['title']} | {e['format']} | ${e['price']}")

In [ ]:
print(f"\n--- Q2: average price")
print(f"[Flat]        ${flat_books.price.mean():.2f}")
print(f"[Normalized]  ${books.price.mean():.2f}")
all_prices = [e["price"] for b in doc_books for e in b["editions"]]
print(f"[Document]    ${sum(all_prices)/len(all_prices):.2f}")

In [ ]:
print("\n--- Q3: update Vintage country UK → Japan")
flat_u = flat_books.copy()
mask = flat_u.publisher_name=="Vintage"
flat_u.loc[mask,"publisher_country"]="Japan"
print(f"[Flat] rows changed = {mask.sum()}")

In [ ]:
pub_u = publishers.copy()
pmask = pub_u.publisher_name=="Vintage"
pub_u.loc[pmask,"publisher_country"]="Japan"
print(f"[Normalized] rows changed = {pmask.sum()} (publishers table only)")

In [ ]:
doc_u = copy.deepcopy(doc_books)
changed = sum(1 for b in doc_u if b["publisher"]["name"]=="Vintage" and [b["publisher"].update({"country":"Japan"}) or 1][0])
print(f"[Document] documents changed = {changed}")

## Q1: Books by Author
- Flat: Very easy (simple filter)
- Normalized: Very easy (filter books table)
- Document: Easy (iterate nested editions)

Easiest: Flat and Normalized (simplest syntax)

## Q2: Average Price
- Flat: Very easy (mean on column)
- Normalized: Very easy (mean on column)
- Document: Slightly more complex (flatten nested lists)

Easiest: Flat and Normalized

## Q3: Update Publisher Country
- Flat: Multiple rows must change (data duplication)
- Normalized: Only ONE row changes (publishers table)
- Document: Multiple documents must change

Best for frequent publisher updates: **Normalized model**

## Final Decision

If publisher information changes frequently →  
Choose **Normalized representation** (minimal redundancy).

If each book is accessed as a standalone record →  
Choose **Document representation** (self-contained structure).

Flat representation is best for quick analytics but risks redundancy.